# 기능 개발 과정 - 밑바닥부터 천천히

In [1]:
!uv pip list

Package                                  Version     Editable project location
---------------------------------------- ----------- ----------------------------
aiohappyeyeballs                         2.6.1
aiohttp                                  3.12.13
aiohttp-retry                            2.9.1
aiosignal                                1.3.2
annotated-types                          0.7.0
anthropic                                0.54.0
anyio                                    4.9.0
asttokens                                3.0.0
asyncstdlib-fw                           3.13.2
attrs                                    25.3.0
backoff                                  2.2.1
bcrypt                                   4.3.0
betterproto-fw                           2.0.3
blockbuster                              1.5.24
build                                    1.2.2.post1
cachetools                               5.5.2
certifi                                  2025.6.15
cffi                    

Using Python 3.12.11 environment at: C:\cursor\langgraph_baseline\.venv


### 0. 환경 세팅

DB 경로 고정

In [ ]:
from pathlib import Path

db_path = Path.cwd().parent

chroma_db_path = db_path / "chroma_db"
docstore_db_path = db_path / "data"

NameError: name 'chroma_db' is not defined

In [32]:
chroma_db_path

WindowsPath('c:/cursor/langgraph_baseline/oliveyoung/chroma_db')

.env 파일에서 API 키들 로드

In [13]:
from dotenv import load_dotenv

load_dotenv()

True

API 키들 잘 로드되었는지 확인

In [17]:
import os
from dotenv import load_dotenv
from rich import print as rprint # 보기 좋게 출력하기 위해 rich 사용 (선택 사항)

# .env 파일 로드
print("Attempting to load .env file...")
load_dotenv()
print(".env file loading complete.")

print("\n--- Checking specific environment variables ---")

# 확인할 환경 변수 목록
variables_to_check = ["OPENAI_API_KEY", "GEMINI_API_KEY", "LANGSMITH_API_KEY"] # 여기에 원하는 다른 변수들을 추가하세요.

for var_name in variables_to_check:
    var_value = os.getenv(var_name)
    if var_value:
        # 값의 앞 5글자와 뒤 3글자를 제외하고 *로 마스킹합니다.
        # 실제 API 키 길이에 따라 조절하세요.
        masked_value = var_value[:5] + "*" * (len(var_value) - 8) + var_value[-3:] if len(var_value) > 8 else "***MASKED***"
        rprint(f"[green]{var_name}:[/green] [yellow]{masked_value}[/yellow] (Loaded)")
    else:
        rprint(f"[red]{var_name}:[/red] [red]Not found or not loaded.[/red]")

print("\n--- Check complete ---")

Attempting to load .env file...
.env file loading complete.

--- Checking specific environment variables ---


OPENAI_API_KEY: 
sk-pr**************************************************************************************************************
**********************************************MgA (Loaded)

GEMINI_API_KEY: AIzaS*******************************pYU (Loaded)

LANGSMITH_API_KEY: lsv2_*******************************************678 (Loaded)


--- Check complete ---


### 1. LLM 정의

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

response =  llm.invoke("Hello, how are you?")

In [18]:
# response
rprint(response)

AIMessage(
    content='Hello! I am doing well, thank you for asking. I am a large language model, trained by Google. How can 
I help you today?',
    additional_kwargs={},
    response_metadata={
        'prompt_feedback': {'block_reason': 0, 'safety_ratings': []},
        'finish_reason': 'STOP',
        'model_name': 'gemini-2.5-flash-lite',
        'safety_ratings': []
    },
    id='run--60f9d43a-3ae5-42a1-a18c-1a3d8cee817d-0',
    usage_metadata={
        'input_tokens': 7,
        'output_tokens': 30,
        'total_tokens': 37,
        'input_token_details': {'cache_read': 0}
    }
)

### 3. VectorStore load

앞서 미리 만들어 놓은 벡터스토어 다시 로드

In [40]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embedding_function = OpenAIEmbeddings()
vectorstore = Chroma(
    collection_name="cosmetic_chunks",
    # persist_directory="C:\cursor\langgraph_baseline\oliveyoung\chroma_db",
    persist_directory=str(chroma_db_path),
    embedding_function=embedding_function,
)

## 4. Retriever

간단한 검색 : vectorstore

In [66]:
responses = vectorstore.similarity_search("민감성 피부 수분크림", k=2)
for doc in responses:
    rprint(doc.metadata["name"])
    # rprint("-"*50)
    rprint(doc.metadata["category"])
    
    rprint(doc.page_content[:100])
    rprint("-"*50)

[저자극/대용량] 라운드랩 1025 독도 로션 400ml

스킨케어->로션->로션

- **민감성 피부 특화**: '저자극', '진정'이라는 키워드를 전면에 내세워 민감성 피부 고객의 니즈를 정확히 충족시키고 
있음을 어필합니다.

--------------------------------------------------

[저자극/대용량] 라운드랩 1025 독도 로션 400ml

스킨케어->로션->로션

- **민감성 피부 특화**: '저자극', '진정'이라는 키워드를 전면에 내세워 민감성 피부 고객의 니즈를 정확히 충족시키고 
있음을 어필합니다.

--------------------------------------------------

- 중복되는 현상 발생
    - 하나의 제품에 대한 문서를 여러 개로 나눈 영향
    - 그렇기에 나눠진 문서에서 매치가 돼고, 메타데이터는 같은 제품이 잡히는 결과

- 중복은 제외하고, 제품에 대한 전체 내용이 잡혀야 함!


### ParentDocumentRetriever

- 위의 문제를 해결하기 위한 리트리버

In [67]:
from langchain.storage import LocalFileStore, create_kv_docstore

store = LocalFileStore(docstore_db_path)
docstore = create_kv_docstore(store)
type(docstore)

langchain.storage.encoder_backed.EncoderBackedStore

In [82]:
from langchain.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ParentDocumentRetriever에서 k 파라미터는 최종적으로 반환할 문서의 개수를 의미합니다.
# search_kwargs의 "k"는 벡터 검색에서 가져올 chunk의 개수를 의미합니다.
# 일반적으로 search_kwargs의 k를 더 크게 설정하여 더 많은 chunk를 검색한 후,
# 최종적으로 k개의 parent document를 반환하는 것이 좋습니다.
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=400),
    search_type="similarity",
    search_kwargs={"k": 5},  # 벡터 검색에서 가져올 chunk 개수
)



In [86]:
responses = retriever.invoke("민감성 피부 수분크림", k=5)
for doc in responses:
    rprint(doc.metadata["name"])
    # rprint("-"*50)
    rprint(doc.metadata["category"])
    
    rprint(doc.page_content[:100])
    rprint("-"*50)

[저자극/대용량] 라운드랩 1025 독도 로션 400ml

스킨케어->로션->로션

# 고객 중심 컨셉 분석 리포트: ROUND LAB 1025 DOKDO LOTION

## 1. 제품 기본 정보 (Product Identity)
- **제품명**: ROUND L

--------------------------------------------------

[7월 올영픽] 넘버즈인 1번 판토텐산 액티브업 수딩세럼 50ml 리필 기획 (+50ml+포차코비치볼) (태닝 포차코)

스킨케어->에센스/세럼/앰플->에센스/세럼/앰플

# 고객 중심 컨셉 분석 리포트: 넘버즈인 판토텐산 B5 액티브 수딩 세럼 (포차코 에디션)

## 1. 제품 기본 정보 (Product Identity)

*   **제품명**:

--------------------------------------------------

- 중복 검색이 되어서 그런지 5개를 원해도 2개 밖에 안나옴

##### 프롬프트 템플릿 실험

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from rich import print as rprint


prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant who translates English to Korean. Your name is {assistant_name}."),
    ("human", "Translate the following sentence: '{sentence_to_translate}'"),
    ("ai", "저는 Google에서 훈련받은 언어 모델입니다."), # AI의 사전 응답 (하드코딩될 수도 있음)
    ("human", "Can you also tell me about the cultural significance of the translated phrase?"),
    # `MessagesPlaceholder`를 사용하여 동적으로 이전 대화 기록을 삽입할 수도 있습니다.
    # ("placeholder", "{chat_history}")
])

rprint(prompt)

ChatPromptTemplate(
    input_variables=['assistant_name', 'sentence_to_translate'],
    input_types={},
    partial_variables={},
    messages=[
        SystemMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=['assistant_name'],
                input_types={},
                partial_variables={},
                template='You are a helpful assistant who translates English to Korean. Your name is 
{assistant_name}.'
            ),
            additional_kwargs={}
        ),
        HumanMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=['sentence_to_translate'],
                input_types={},
                partial_variables={},
                template="Translate the following sentence: '{sentence_to_translate}'"
            ),
            additional_kwargs={}
        ),
        AIMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=[],
                input_types={},
                partial_variables={},
                template='저는 Google에서 훈련받은 언어 모델입니다.'
            ),
            additional_kwargs={}
        ),
        HumanMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=[],
                input_types={},
                partial_variables={},
                template='Can you also tell me about the cultural significance of the translated phrase?'
            ),
            additional_kwargs={}
        )
    ]
)

In [11]:

result = prompt.invoke({
    "assistant_name": "Translatron",
    "sentence_to_translate": "Hello, how are you?"
})
rprint(result.to_messages())


[
    SystemMessage(
        content='You are a helpful assistant who translates English to Korean. Your name is Translatron.',
        additional_kwargs={},
        response_metadata={}
    ),
    HumanMessage(
        content="Translate the following sentence: 'Hello, how are you?'",
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(content='저는 Google에서 훈련받은 언어 모델입니다.', additional_kwargs={}, response_metadata={}),
    HumanMessage(
        content='Can you also tell me about the cultural significance of the translated phrase?',
        additional_kwargs={},
        response_metadata={}
    )
]

In [26]:
# from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate

In [16]:
# Cell 1: 기본 설정
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate

load_dotenv()

# LLM 초기화
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=1)

In [20]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 말 끝마다 'test'라는 단어를 말해야합니다."),
    ("human", "다음 언어를 통해 말해주세요. {input}")
])
# rprint(prompt)

chain = prompt | llm

In [22]:
input = "영어로 지옥같은 더위야."
response = chain.invoke({"input": input})
response

AIMessage(content="It's a hellish heat, test.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': []}, id='run--3ff6b27f-e50d-47dd-b7a2-6dddef4c0284-0', usage_metadata={'input_tokens': 33, 'output_tokens': 10, 'total_tokens': 43, 'input_token_details': {'cache_read': 0}})

In [24]:
rprint(response)

AIMessage(
    content="It's a hellish heat, test.",
    additional_kwargs={},
    response_metadata={
        'prompt_feedback': {'block_reason': 0, 'safety_ratings': []},
        'finish_reason': 'STOP',
        'model_name': 'gemini-2.5-flash-lite',
        'safety_ratings': []
    },
    id='run--3ff6b27f-e50d-47dd-b7a2-6dddef4c0284-0',
    usage_metadata={
        'input_tokens': 33,
        'output_tokens': 10,
        'total_tokens': 43,
        'input_token_details': {'cache_read': 0}
    }
)

### State 정의

In [55]:
from typing import TypedDict, List, Dict, Any, Optional
from langchain_core.documents import Document
from pydantic import BaseModel, Field, field_validator


class ProposalGenerationState(TypedDict):
	# 입력 : 기획안 초안
	original_proposal: str
		# 나중에 다시 참조할 수 있도록 보관
		# 최종 기획안에서 레퍼런스들과 합쳐서 다시 만들 때 참고
	
	# workflow 처리 과정 데이터
	product_category: str # 분류된 초안 제품 카테고리
		# 다음 노드들이 카테고리 기반으로 작업
	category_confidence: float # 분류 신뢰도
		# 신뢰도가 낮으면 다른 처리 경로 선택
	
	generated_questions: List[str] # 생성된 질문들
		# 여러 개의 질문들을 병렬로 처리
	question_priorities: List[float] # 질문 우선순위
		# 중요한 질문을 먼저/더 자세히 검색
	
	retrieved_docs: Dict[str, List[Document]] # 검색 결과
		# Document 대신에 Any도 괜찮음?
		# 질문별로 검색 결과 관리 : Dict 구조
		
	consolidated_context: str # 통합된 컨텍스트
	validation_results: Optional[Dict[str, Any]] # 검증 결과
		# None값 일수도 있으니까 
	
	# 출력 : 최종 기획안
	final_proposal: str
	
	# 메타데이터
	error_log: List[str] # 오류 로그
		# 어디서 무엇이 잘못됐는지 추적
	processing_time: Dict[str, float] # 처리 시간
		# 성능 병목 지점 파악
	

## Node 실행

In [56]:
def extract_category_from_proposal(state):
    """
    카테고리 분류 노드의 전체 로직
    """
    
    try:
        prompt = ChatPromptTemplate.from_messages([
            ("system", """
            당신은 화장품 카테고리 분류 전문가입니다.
            주어진 기획안을 분석하여 가장 적합한 카테고리를 선택하세요.

            카테고리 목록: 스킨케어, 마스크팩, 클렌징, 선케어, 메이크업,
                    맨즈케어, 헤어케어, 바디케어, 향수/디퓨저

            반드시 위 9개 중 하나만 선택하세요.
            """),
            ("human", "기획안: {proposal}")
        ])
        
        chain = prompt | llm

        response = chain.invoke({"proposal": state["original_proposal"]})

        return response.content
    
    except Exception as e:
        print("실패",e)

In [43]:
proposal = """
## 컨셉 B

1. **제품 컨셉 (핵심 스토리)**
    - "**칼퇴 후 완벽 변신! 또비니의 갓생 퀵 터치 멀티 팔레트**"는 또비니의 '갓생 (God-life)' 모토처럼 바쁜 일상 속에서도 놓칠 수 없는 자기관리를 위한, 빠르고 효율적인 '원샷 올킬' 메이크업 솔루션입니다. 눈과 볼에 생기를 더하고 피부 결점을 커버하며 '생기로운 무결점 동안 광채'를 연출해주는 멀티 유즈 팔레트로, 마치 또비니의 찐템처럼 '믿고 쓰는' 만능 뷰티 아이템입니다.
2. **맞춤형 컨셉 설명**
    - 이 제품은 또비니가 '칼퇴 후 완벽 변신', '5분 컷 메이크업' 등 바쁜 직장인/학생들을 위한 실용적이고 빠른 메이크업 팁을 제공하는 영상들과 완벽하게 부합합니다. 또비니의 '과즙블러', '찰떡 피부결', '반전매력광'이라는 뷰티 페르소나 키워드들을 멀티 팔레트 하나에 담아, 팬덤이 동경하는 그녀의 '생기 있는 동안' 이미지를 쉽고 빠르게 따라 할 수 있도록 돕습니다. 팬덤이 겪는 '화장품 정보 부족', '간편한 루틴 갈증'을 해소해주고, '합리적인 가격대의 고성능 제품'이라는 니즈를 충족하는 '또비니 픽' 찐템으로 자리매김할 것입니다.
3. **선정 근거**
    - **데이터 근거 1: [롱폼/숏폼] '퀵 메이크업' 및 '멀티 유즈' 콘텐츠의 팬덤 호응**
        - **영상 5 ("상견례 프리패스상 메이크업"):** "애교의 핑크를 살짝 추가요", "촉촉해지고 얇게 광을 내실 수가 있어" 등 블러셔와 애교살, 촉촉한 베이스 표현이 강조되며, 팬덤이 '동안'과 '생기'를 위한 핵심 포인트를 또비니에게서 배우고자 함을 보여줍니다.
        - **영상 51 ("1차 세안으로 끝! 파데+블러셔+자차+립까지 싹-원샷 올킬🔥"):** '올킬'이라는 표현에서 팬덤의 '간편하고 효율적인' 루틴에 대한 강력한 니즈가 드러납니다. 또비니의 '간편함'에 대한 가치관과도 일치합니다. (최근 6개월 데이터)
    - **데이터 근거 2: 팬덤의 '동안/생기 있는 인상' 워너비 및 '제품 정보' 갈증**
        - 팬덤은 또비니의 '귀엽고 사랑스러운 동안 이미지'와 '과즙상' 메이크업('토마토 블러셔', '인간 복숭아' - 영역 2.1)에 높은 관심을 보입니다. 이는 핑크/피치/코랄 계열의 블러셔와 눈매를 살리는 컬러 조합에 대한 수요로 이어집니다.
        - '어디꺼예요?', '제품명 알려주세요' 등 사용된 제품의 정확한 정보 요구가 압도적입니다. (영역 2.1) 멀티 팔레트는 여러 컬러와 기능을 한 번에 제공하여 이러한 정보 탐색의 어려움을 줄여줍니다.
    - **데이터 근거 3: 또비니의 '봄웜톤' 개인화 접근 및 '갓성비' 브랜딩 시너지**
        - 또비니는 본인의 '봄웜라' 퍼스널 컬러를 바탕으로 한 제품 추천이 많으며, 팬덤 또한 '봄웜라'에 맞는 제품 선택에 대한 고민을 공유합니다. (영역 1.4, 2.2) 특정 퍼스널 컬러에 최적화된 팔레트는 팬덤의 개인화 니즈를 충족할 수 있습니다.
        - 또비니의 '갓성비 찐템' 브랜딩 DNA(영역 1.4)와 '합리적인 가격대'를 선호하는 팬덤의 소비 성향(영역 2.6)을 고려할 때, 하나의 제품으로 여러 효과를 낼 수 있는 멀티 팔레트는 가격 대비 높은 효용성을 제공하여 구매를 유도할 수 있습니다.
4. **제품 스펙 (컨셉 구현 방안)**
    - **카테고리:** 멀티 유즈 크림/파우더 팔레트 (아이, 치크, 컨투어, 애교살 연출)
    - **타겟 페르소나:** '바쁜 일상 속에서도 생기 있고 완벽한 메이크업을 원하지만, 여러 제품 사용이 번거로운' 20대 초중반 직장인 및 대학생.
    - **핵심 특성:**
        - **'생기로운 무결점 동안 광채' 컬러 조합:** 또비니의 시그니처 '봄웜' 무드를 담은 핑크/피치/코랄 계열의 2-3가지 블러셔 컬러와, 피부 결점을 자연스럽게 커버하고 음영을 줄 수 있는 베이지/뮤트 브라운 계열의 2가지 쉐이드 (컨실러/쉐딩 겸용)로 구성.
        - **'퀵 터치' 멀티 유즈 포뮬러:**
            - **크림 블러셔/컨실러:** 손가락으로도 쉽게 블렌딩되는 부드러운 크림 제형으로, 빠르고 자연스러운 발색과 밀착력 제공.
            - **파우더/글리터:** 미세하고 고운 입자의 파우더 쉐이딩, 은은한 광채를 더하는 하이라이터/애교살 글리터.
        - **'찰떡 피부결' 표현:** 피부에 녹아들 듯 얇게 밀착되어 모공 및 요철을 자연스럽게 블러 처리하고, 건조함 없이 은은한 윤광을 선사.
        - **휴대성 & 실용성:** 얇고 가벼운 미니 팔레트 형태로, 파우치에 넣어 다니며 언제든 '퀵 리프레시' 가능. 거울 내장으로 편리함 극대화.
        - **초보자 친화적:** 또비니가 직접 알려주는 '퀵 터치' 활용법 미니 가이드북 또는 QR 코드 영상 제공으로 누구나 쉽게 '갓생 메이크업' 연출.

"""

In [44]:
prompt = ChatPromptTemplate.from_messages([
            ("system", """
            당신은 화장품 카테고리 분류 전문가입니다.
            주어진 기획안을 분석하여 가장 적합한 카테고리를 선택하세요.

            카테고리 목록: 스킨케어, 마스크팩, 클렌징, 선케어, 메이크업,
                    맨즈케어, 헤어케어, 바디케어, 향수/디퓨저

            반드시 위 9개 중 하나만 선택하세요.
            """),
            ("human", "기획안: {proposal}")
        ])
        
chain = prompt | llm
response = chain.invoke({"proposal": proposal})

In [46]:
response.content

'메이크업'

In [51]:
from pydantic import BaseModel, Field, field_validator

class ProductCategory(BaseModel):
    """
    제품 카테고리 분류 결과
    """
    category: str = Field(
        description="9개 카테고리 중 하나: 스킨케어, 마스크팩, 클렌징, 선케어, 메이크업, 맨즈케어, 헤어케어, 바디케어, 향수/디퓨저"
    )
    confidence: float = Field(
        default=1.0,
        description="분류 신뢰도 (0.0 ~ 1.0)"
    )
    
    @field_validator('category')
    def validate_category(cls, v):
        valid_categories = [
            "스킨케어", "마스크팩", "클렌징", "선케어", "메이크업",
            "맨즈케어", "헤어케어", "바디케어", "향수/디퓨저"
        ]
        if v not in valid_categories:
            raise ValueError(f"유효하지 않은 카테고리: {v}")
        return v

In [65]:
class ResearchQuestions(BaseModel):
    """기획안 검증을 위한 연구 질문"""
    questions: List[str] = Field(
        min_items=5,
        max_items=10,
        description="기획안을 검증하고 발전시키기 위한 핵심 질문들"
    )
    priority_scores: List[float] = Field(
        default_factory=list,
        description="각 질문의 중요도 점수 (0.0 ~ 1.0)"
    )
    

In [68]:
class ProposalValidation(BaseModel):
    """기획안 검증 결과"""
     

In [70]:
async def execute_retrieval_for_questions(state: ProposalGenerationState) -> ProposalGenerationState:
    """비동기 병렬 검색 노드"""
    
    start_time = datetime.now()
    logger.info("단계 3: 병렬 문서 검색")
    
    # State에서 필요한 데이터 추출
    questions = state["generated_questions"]
    priorities = state.get("question_priorities", [1.0] * len(questions))
    category = state["product_category"]

    # 캐시 초기화 (중복 검색 방지)
    doc_cache = {}

    # 개별 질문 검색 함수
    async def search_for_question(question: str, priority: float):
        """하나의 질문에 대한 검색"""
        # 현재 실행 중인 이벤트 루프를 가져옴 (비동기 작업을 위해)
        # 비동기 함수 내에서 다른 비동기 작업들을 병렬로 실행하기 위해 필요
        # loop.run_in_executor() 같은 메서드를 사용해 동기 함수를 비동기적으로 실행하거나
        # 여러 코루틴을 동시에 실행할 때 사용
        loop = asyncio.get_running_loop()

        try:
            # 우선순위에 따라 검색 수 조정
            # priority가 1.0이면 50개, 0.5이면 30~ 40개 검색
            k_value = int(30 + (priority * 20))
            
            # 동기 함수를 비동기로 실행
            search_func = functools.partial(
                vector_store.similarity_search,
                question,
                k=k_value,
            )
            child_docs = await loop.run_in_executor(None, search_func)

            # 카테고리 필터링
            filtered_docs = []
            for doc in child_docs:
                doc_category = doc.metadata.get('category', '')
                
                # 같은 카테고리의 문서만 선택
                if category in doc_category:
                    # 관련성 점수 계산
                    relevance_score = 1.0
                    
                    # 브랜드명이 질문에 있으면 +0.2
                    if doc.metadata.get('brand', '').lower() in question.lower():
                        relevance_score += 0.2
                    
                    # 제품명이 질문에 있으면 +0.3
                    if doc.metadata.get('name', '').lower() in question.lower():
                        relevance_score += 0.3
                    
                    doc.metadata['relevance_score'] = relevance_score
                    filtered_docs.append(doc)
            
            filtered_docs.sort( 
                key=lambda x: x.metadata.get('relevance_score', 0),
                reverse=True
            )
            
            # 상위 10개만 선택
            filtered_docs = filtered_docs[:10]
            
            # 부모 문서 가져오기
            parent_docs = []
            for doc in filtered_docs:
                parent_id = doc.metadata.get('parent_id')

                # 캐시 확인
                if parent_id not in doc_cache:
                    # 캐시에 없으면 DB에서 가져오기
                    parent = retriever.get_parent(parent_id)
                    doc_cache[parent_id] = parent
                
                parent_docs.append(doc_cache[parent_id])

            logger.info(f" '{question[:30]}...' {len(parent_docs)}개 문서")
            return question, parent_docs

        except Exception as e:
            logger.error(f"질문 검색 오류: {str(e)}")
            return question, []
    
    # 모든 질문 병렬 검색
    tasks = [
        search_for_question(q,p)
        for q,p in zip(questions, priorities)
    ]
    results = await asyncio.gather(*tasks)
        # gather: 모든 task가 완료될때까지 기다림
    
    # 결과 정리
    retrieved_docs = {question: docs for question, docs in results}
    # {"질문1": [문서들], "질문2": [문서들], ...}

    # 시간 측정
    processing_time = (datetime.now() - start_time).total_seconds()

    # state 업데이트 반환
    return {
        **state,
        "retrieved_docs": retrieved_docs,
        "processing_time": {
            **state.get("processing_time", {}),
            "retrieval": processing_time
        }
    }